In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer, Input, LSTM, Embedding, Dense, Dropout, Bidirectional
from tensorflow.keras.models import Model
import pickle
import re

In [2]:
df = pd.read_csv(r"C:\Users\Hp\Downloads\My ADL project\IMDB Dataset.csv")
df


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [3]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [4]:
#Feature Engineering: Basic Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<br />', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [5]:
# Tokenization
max_features = 10000
maxlen = 100
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(df['review'])

X = pad_sequences(tokenizer.texts_to_sequences(df['review']), maxlen=maxlen)
y = df['sentiment'].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
class AttentionLayer(Layer):
    def __init__(self, method='additive', **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.method = method

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], input_shape[-1]), initializer="glorot_uniform", trainable=True)
        self.v = self.add_weight(name="att_v", shape=(input_shape[-1], 1), initializer="glorot_uniform", trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        if self.method == 'multiplicative':
            # Luong style: score = h_t * W * h_s
            # For self-attention on single sequence, we align the sequence with itself
            score = tf.matmul(inputs, self.W)
            score = tf.matmul(score, inputs, transpose_b=True)
            # Take the diagonal or mean of scores to get a weight per timestep
            score = tf.reduce_mean(score, axis=-1, keepdims=True)
        else:
            # Bahdanau style: score = v * tanh(W * h)
            score = tf.tanh(tf.tensordot(inputs, self.W, axes=1))
            score = tf.tensordot(score, self.v, axes=1)

        weights = tf.nn.softmax(score, axis=1)
        context = inputs * weights
        return tf.reduce_sum(context, axis=1)



In [7]:
def build_model(attention_type='additive'):
    inputs = Input(shape=(maxlen,))
    x = Embedding(max_features, 128)(inputs)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = AttentionLayer(method=attention_type)(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [8]:
# Training and Comparing
print("Training Multiplicative Attention Model...")
model_mult = build_model('multiplicative')
history_mult = model_mult.fit(X_train, y_train, epochs=2, batch_size=64, validation_split=0.1, verbose=1)

print("\nTraining Additive Attention Model...")
model_add = build_model('additive')
history_add = model_add.fit(X_train, y_train, epochs=2, batch_size=64, validation_split=0.1, verbose=1)

# Save best model and tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save the best performing one (usually additive for this task)
model_add.save('best_model.h5')

Training Multiplicative Attention Model...

Epoch 1/2


c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:857: UserWarning: Gradients do not exist for variables ['attention_layer/att_v'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


563/563 ━━━━━━━━━━━━━━━━━━━━ 90s 147ms/step - accuracy: 0.8184 - loss: 0.3926 - val_accuracy: 0.8605 - val_loss: 0.3320
Epoch 2/2
563/563 ━━━━━━━━━━━━━━━━━━━━ 83s 148ms/step - accuracy: 0.8908 - loss: 0.2679 - val_accuracy: 0.8510 - val_loss: 0.3387

Training Additive Attention Model...
Epoch 1/2
563/563 ━━━━━━━━━━━━━━━━━━━━ 84s 140ms/step - accuracy: 0.8249 - loss: 0.3860 - val_accuracy: 0.8665 - val_loss: 0.3214
Epoch 2/2
563/563 ━━━━━━━━━━━━━━━━━━━━ 106s 189ms/step - accuracy: 0.8951 - loss: 0.2554 - val_accuracy: 0.8575 - val_loss: 0.3322


In [9]:
# Inference and Comparison Results
import matplotlib.pyplot as plt

def predict_sentiment(text):
    # Load inside function to demonstrate standalone capability
    with open('tokenizer.pkl', 'rb') as f:
        tok = pickle.load(f)
    model = tf.keras.models.load_model('best_model.h5', custom_objects={'AttentionLayer': AttentionLayer})

    seq = tok.texts_to_sequences([clean_text(text)])
    pad = pad_sequences(seq, maxlen=maxlen)
    pred = model.predict(pad, verbose=0)[0][0]
    return "Positive" if pred > 0.5 else "Negative", pred



In [10]:
# Project Summary
mult_acc = history_mult.history['val_accuracy'][-1]
add_acc = history_add.history['val_accuracy'][-1]

print(f"Multiplicative Attention Accuracy: {mult_acc:.4f}")
print(f"Additive Attention Accuracy: {add_acc:.4f}")
print(f"Best Attention: {'Additive' if add_acc > mult_acc else 'Multiplicative'}")
print("Best Loss Function: Binary Crossentropy (Standard for binary classification)")
print("Best Optimizer: Adam (Adaptive learning rate for LSTMs)")

# Test Inference
test_review = "The cinematography was breathtaking and the plot was engaging."
label, score = predict_sentiment(test_review)
print(f"\nSample Review: {test_review}")
print(f"Inference Result: {label} (Confidence: {score:.4f})")

Multiplicative Attention Accuracy: 0.8510
Additive Attention Accuracy: 0.8575
Best Attention: Additive
Best Loss Function: Binary Crossentropy (Standard for binary classification)
Best Optimizer: Adam (Adaptive learning rate for LSTMs)



Sample Review: The cinematography was breathtaking and the plot was engaging.
Inference Result: Negative (Confidence: 0.4512)
